# ICARUS baselines: VAE / CAE

Uses checkpoints from `train/04_TrainAutoencoders_ICARUS.ipynb`
(or any EMA under `DATA_ROOT/training/{vae,cae}/icarus/`).

Compare reconstruction residuals on ICARUS DNN-ROI `.h5` tiles and/or
legacy defect NPZs if present. Default scoring: mean-squared residual
and top-tile RMS (mirrors SBND demos).

**GPU:** keep `DEVICE = "cpu"` until approved.


In [ ]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as th

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
sys.path.insert(0, str(APP_ROOT))
sys.path.insert(0, str(APP_ROOT / "train" / "diffusion-anomaly"))
if "configs.paths" in sys.modules:
    importlib.reload(sys.modules["configs.paths"])

from configs.paths import (
    DATA_ROOT,
    GPUTNAM_DNN_ROI,
    ae_model_config,
    default_ae_checkpoint,
    describe_environment,
    ensure_layout,
)
from guided_diffusion.image_datasets import load_image_file
from guided_diffusion.script_util import create_autoencoder

ensure_layout()
print(json.dumps(describe_environment(), indent=2))

OUT = DATA_ROOT / "inference" / "baselines" / "icarus"
FIG = DATA_ROOT / "figures" / "baselines_icarus_vae_cae"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"
AE_TYPES = ["vae", "cae"]
NUM_TILES = 4
NUM_FILES = 2
SEED = 0


In [ ]:
ckpts = {t: default_ae_checkpoint(t, "icarus") for t in AE_TYPES}
for t, p in ckpts.items():
    print(f"{t}: {p}")

h5_files = sorted(GPUTNAM_DNN_ROI.glob("*.h5"))[:NUM_FILES] if GPUTNAM_DNN_ROI.is_dir() else []
print(f"ICARUS files ({len(h5_files)}):", [p.name for p in h5_files])
if not h5_files:
    raise FileNotFoundError(f"No ICARUS DNN-ROI h5 under {GPUTNAM_DNN_ROI}")


In [ ]:
def load_model(ae_type: str, ckpt: Path, device: th.device):
    cfg = ae_model_config(ae_type)
    model = create_autoencoder(**cfg)
    model.load_state_dict(th.load(ckpt, map_location="cpu", weights_only=True))
    return model.to(device).eval()


def anomaly_on_file(model, path: Path, num_tiles: int, device: th.device):
    images, _ = load_image_file(path, 512)
    order = np.argsort(-np.abs(images).sum(axis=(1, 2, 3)))[:num_tiles]
    inp = th.from_numpy(images[order]).to(device)
    with th.no_grad():
        reco = model.reconstruct(inp)
    sal = (inp - reco).cpu().numpy()
    inp_np, reco_np = inp.cpu().numpy(), reco.cpu().numpy()
    scores = [float(np.mean(sal[k] ** 2)) for k in range(len(order))]
    return order, inp_np, reco_np, sal, scores


device = th.device(DEVICE)
summary = []
for ae in AE_TYPES:
    ckpt = ckpts[ae]
    if ckpt is None:
        print(f"skip {ae}: train first via train/04_TrainAutoencoders_ICARUS.ipynb")
        continue
    model = load_model(ae, ckpt, device)
    ae_out = OUT / ae
    ae_out.mkdir(parents=True, exist_ok=True)
    for path in h5_files:
        order, inp, reco, sal, scores = anomaly_on_file(model, path, NUM_TILES, device)
        stem = path.stem
        np.savez_compressed(
            ae_out / f"{stem}_anomaly.npz",
            input=inp, reco=reco, saliency=sal, tile_index=order, source=str(path),
        )
        for k, tile in enumerate(order):
            fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
            for ax, arr, title, kw in (
                (axes[0], inp[k, 0], f"Original tile {tile}", dict(vmin=-1, vmax=1)),
                (axes[1], reco[k, 0], "Reconstruction", dict(vmin=-1, vmax=1)),
                (axes[2], sal[k, 0], "Saliency", dict(cmap="bwr", vmin=-0.1, vmax=0.1)),
            ):
                im = ax.imshow(arr, **kw)
                ax.set_title(title)
                fig.colorbar(im, ax=ax, fraction=0.046)
            fig.suptitle(f"{ae.upper()} | {stem} | mse={scores[k]:.3e}")
            fig.savefig(FIG / f"{ae}_{stem}_tile{k:02d}.png", bbox_inches="tight")
            plt.close(fig)
            summary.append({"ae": ae, "file": stem, "tile": int(tile), "mse": scores[k]})
        print(ae, stem, "scores", scores)

import pandas as pd
df = pd.DataFrame(summary)
display(df)
if not df.empty:
    display(df.groupby("ae")["mse"].describe())
    (OUT / "icarus_baseline_summary.json").write_text(df.to_json(orient="records", indent=2))
